<a href="https://colab.research.google.com/github/Sanath2701/Car-predictor/blob/main/Price_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install requests beautifulsoup4 pandas streamlit pyngrok -q

import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import os
from pyngrok import ngrok, conf

def scrape_ebay(search_query):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
    }
    url = f"https://www.ebay.com/sch/i.html?_nkw={search_query.replace(' ', '+')}"
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, 'html.parser')
    items = soup.select('.s-item')
    products = []
    for item in items:
        title_tag = item.select_one('.s-item__title')
        price_tag = item.select_one('.s-item__price')
        if title_tag and price_tag:
            title = title_tag.text
            price = price_tag.text
            products.append({'Title': title, 'Price': price})
    df = pd.DataFrame(products)
    return df

search_query = "shoes"
df = scrape_ebay(search_query)
df.to_csv('products.csv', index=False)

streamlit_code = '''
import streamlit as st
import pandas as pd

df = pd.read_csv('products.csv')

st.set_page_config(page_title="eBay Product Search", layout="centered")
st.title('🔎 eBay Product Search Dashboard')

search_term = st.text_input('Enter product name:')

if st.button('Search'):
    if search_term:
        results = df[df['Title'].str.contains(search_term, case=False, na=False)]
        if not results.empty:
            st.success(f'Found {len(results)} results:')
            st.dataframe(results)
        else:
            st.warning('No products found.')
    else:
        st.info('Please enter a product name to search.')
'''

with open('app.py', 'w') as f:
    f.write(streamlit_code)

!ngrok config add-authtoken 2wJvSXVw3PcjrW87bhMbODYLwoG_4TtgQhQ1VzE4DZScmTyGo
!pkill streamlit
!streamlit run app.py &>/content/logs.txt &
time.sleep(5)
conf.get_default().region = "us"
public_url = ngrok.connect(addr="http://localhost:8501", proto="http")
print(f'{public_url}')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 817.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 2.1 MB/s eta 0:00:00
Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
NgrokTunnel: "https://3663-34-82-137-86.ngrok-free.app" -> "http://localhost:8501"
